## Part 3 of 6: Exploratory Data Analysis

Loads df, RAW_DOLLAR_COLS, and failed_firms from notebooks/02_data_cleaning.ipynb.

See `notebooks/README.md` for the full run order.

In [ ]:
# Import Python libraries

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (roc_auc_score, average_precision_score, f1_score, precision_recall_curve, precision_score, recall_score)
import warnings
warnings.filterwarnings("ignore")
np.random.seed(42)
print("Libraries have been imported!")

In [ ]:
# Load state saved by the previous notebook
import joblib
_state = joblib.load("_state/02_state.joblib")
globals().update(_state)
print(f"Loaded {len(_state)} objects: {sorted(_state)}")


In [ ]:
# Graph distribution of target variable
# Also graph the 'corrected' target variable that only counts the last year
raw_dist = df["status_label"].value_counts(normalize=True)
true_rate = df["target"].mean()
print("Raw row-level label distribution:")
print(raw_dist.round(4).to_string())
print(f"Corrected positive rate after relabel: {true_rate:.4%} "
      f"({df['target'].sum()} of {len(df):,} rows)")

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
raw_dist.plot(kind="bar", ax=axes[0], color=["#4C72B0", "#C44E52"], rot=0)
axes[0].set_title("Raw target variable distribution (pre-cleaning)")
axes[0].set_ylabel("Proportion of rows")
pd.Series({"alive next yr": 1 - true_rate, "bankrupt next yr": true_rate}) \
    .plot(kind="bar", ax=axes[1], color=["#4C72B0", "#C44E52"], rot=0)
axes[1].set_title("Target variable distribution")
plt.tight_layout(); plt.show()

In [ ]:
# Graph Bankruptcy rate by year, counts by split
# Note: Data year X = the year BEFORE filing,
# so a spike at 2007-2008 corresponds to 2008-2009 filings.
by_year = df.groupby("year").agg(positives=("target", "sum"),
                                 rows=("target", "size"))
by_year["rate"] = by_year["positives"] / by_year["rows"]

fig, ax1 = plt.subplots(figsize=(10, 4))
ax1.plot(by_year.index, by_year["rate"] * 100, marker="o", color="#C44E52")
ax1.set_xlabel("Data year (filing occurs the following year)")
ax1.set_ylabel("% of firm-years bankrupt next year", color="#C44E52")
ax1.axvspan(2007, 2008.5, alpha=0.15, color="gray",
            label="2008-09 filings window")
ax1.set_title("Bankruptcy rate by year")
ax1.legend()
plt.tight_layout(); plt.show()

# Build training/validation/testing splits
splits = {"train (1999-2011)": (1999, 2011),
          "val   (2012-2014)": (2012, 2014),
          "test  (2015-2018)": (2015, 2018)}
print("Class counts by split window:")
for name, (lo, hi) in splits.items():
    m = df["year"].between(lo, hi)
    print(f"  {name}: rows={m.sum():>6,}  positives={int(df.loc[m,'target'].sum()):>4}"
          f"  rate={df.loc[m,'target'].mean():.3%}")

In [ ]:
# Graph feature distributions by class
# Use a signed-log x-axis so heavy tails don't flatten the comparison visually.
EDA_features = ["current_assets", "cogs", "ebitda", "net_income",
             "dep_amort", "inventory", "receivables", "total_assets"]
slog = lambda s: np.sign(s) * np.log1p(np.abs(s))
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
for ax, col in zip(axes.ravel(), EDA_features):
    sns.kdeplot(slog(df.loc[df["target"] == 0, col]), ax=ax, label="alive",
                fill=True, alpha=0.4)
    sns.kdeplot(slog(df.loc[df["target"] == 1, col]), ax=ax, label="bankrupt next year",
                fill=True, alpha=0.4, color="#C44E52")
    ax.set_title(col); ax.set_xlabel("signed log scale")
axes[0, 0].legend()
fig.suptitle("Raw feature distributions by class", y=1.02)
plt.tight_layout(); plt.show()

In [ ]:
# Heatmap of raw-feature correlations
corr_raw = df[RAW_DOLLAR_COLS].corr()
plt.figure(figsize=(11, 8))
sns.heatmap(corr_raw, cmap="coolwarm", center=0, annot=False)
plt.title("Raw dollar features: Pearson correlation (diagnostic only)")
plt.tight_layout(); plt.show()
high = [(a, b, corr_raw.loc[a, b]) for i, a in enumerate(RAW_DOLLAR_COLS)
        for b in RAW_DOLLAR_COLS[i + 1:] if abs(corr_raw.loc[a, b]) > 0.85]
print(f"{len(high)} raw pairs exceed r =|0.85|, likely driven by firm size.")

In [ ]:
# Plot the bankruptcy trajectory of specific example firms, to see what happened
hist_len = df[df["company_name"].isin(failed_firms)] \
    .groupby("company_name")["year"].count()
traj_firms = hist_len[hist_len >= 6].sample(8, random_state=42).index
fig, axes = plt.subplots(2, 4, figsize=(16, 7), sharex=False)
for ax, firm in zip(axes.ravel(), traj_firms):
    g = df[df["company_name"] == firm]
    for col, c in [("ebitda", "#4C72B0"), ("net_income", "#C44E52"),
                   ("current_assets", "#55A868")]:
        ax.plot(g["year"], g[col], marker=".", label=col, color=c)
    ax.axhline(0, color="gray", lw=0.8)
    ax.axvline(g["year"].max(), color="black", ls="--", lw=0.8)
    ax.set_title(f"{firm} (files {g['year'].max() + 1})", fontsize=9)
axes[0, 0].legend(fontsize=7)
fig.suptitle("Deterioration trajectories of example failed firms", y=1.02)
plt.tight_layout(); plt.show()

In [ ]:
# Analysis of outliers
pct = df[RAW_DOLLAR_COLS].quantile([0.01, 0.99]).T
pct.columns = ["p01", "p99"]
print("1st/99th percentiles of raw features:")
print(pct.round(1).to_string())
fig, ax = plt.subplots(figsize=(12, 4))
sns.boxplot(data=df[RAW_DOLLAR_COLS].apply(slog), ax=ax)
ax.set_title("Boxplots of each feature on signed-log scale")
plt.xticks(rotation=45, ha="right")
plt.tight_layout(); plt.show()

In [ ]:
# Save state for the next notebook
import joblib
from pathlib import Path
Path("_state").mkdir(exist_ok=True)
joblib.dump({
    "df": df
}, "_state/03_state.joblib")
